In [ ]:
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats
import pandas as pd
from tqdm import tqdm

# Load the shapefile
ecoregions = gpd.read_file("G:/Hangkai/CONUS Forest Edge Mapping/CONUS shapefile/CONUS_ECOSYSTEM.shp")

# Assuming you are working with two years for now, adjust as necessary
years = [2001, 2002]
tif_paths = [f"G:/Hangkai/CONUS Forest Edge Mapping/CONUS Forest Edge/{year}LC_edges.tif" for year in years]


for tif_path in tif_paths:
    with rasterio.open(tif_path) as src:
        tif_crs = src.crs
        # Check and harmonize CRS
        print(ecoregions.crs)
        print(src.crs)
        
        if ecoregions.crs != tif_crs:
            # Reproject ecoregions to match the TIFF CRS
            ecoregions = ecoregions.to_crs(tif_crs)

# Proceed with your analysis as before
all_data = []

# Loop over each year and TIFF file
for year, tif_path in zip(years, tif_paths):
    with rasterio.open(tif_path) as src:
        nodata = src.nodata
        if nodata is None:
            nodata = -999  # Define a default nodata value if not specified in the raster

        # Calculate zonal statistics for each ecoregion
        stats = zonal_stats(ecoregions, src.read(1), affine=src.transform, stats='count', categorical=True, nodata=nodata)

        # Iterate over each ecoregion and its statistics
        for ecoregion, stat in zip(ecoregions.itertuples(), stats):
            ecoregion_name = getattr(ecoregion, 'NA_L1NAME')
            categorical_data = stat.get('categorical', {})
            
            # Iterate over each possible edge type from 0 to 15
            for edge_type in range(16):
                count = categorical_data.get(edge_type, 0)  # Get the count for each edge type, default to 0 if not found
                all_data.append({
                    'ecoregion': ecoregion_name,
                    'year': year,
                    'edge_type': edge_type,
                    'count': count
                })


df = pd.DataFrame(all_data)
csv_path = "G:/Hangkai/CONUS Forest Edge Mapping/CONUS Forest Edge/edge_statistics_test.csv"
df.to_csv(csv_path, index=False)

print(f"Saved edge statistics to {csv_path}")
